# Setup FDSN client

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

t0 = UTCDateTime("2009-03-20T00:00:00")
t1 = UTCDateTime("2009-03-23T00:00:00")

# Redoubt centre: 60°29′06″N 152°44′31″W
REDOUBT_LAT = 60 + 29/60 + 6/3600
REDOUBT_LON = -(152 + 44/60 + 31/3600)

EVENT_RADIUS_DEG = 0.4
STATION_RADIUS_DEG = 0.2

# Create a DATA/Redoubt_2009 directory from your HOME directory

In [ ]:
import os
home_dir = os.path.expanduser("~")
output_dir = os.path.join(home_dir, "DATA", "Redoubt_2009")
os.makedirs(output_dir, exist_ok=True)


# Get events

In [ ]:
usgsclient = Client("USGS")  # queries earthquake.usgs.gov FDSN event service  [oai_citation:3‡USGS](https://earthquake.usgs.gov/fdsnws/event/1/?utm_source=chatgpt.com)

'''
cat = usgs.get_events(
    starttime=t0, endtime=t1,
    minlatitude=60.2, maxlatitude=60.8,
    minlongitude=-153.2, maxlongitude=-152.2,
    # optionally: minmagnitude=1.5,
)
'''

cat = usgsclient.get_events(
    starttime=t0, endtime=t1,
    latitude=REDOUBT_LAT, longitude=REDOUBT_LON,
    maxradius=EVENT_RADIUS_DEG,
    #catalog="ak",
    includeallorigins=True,
    includeallmagnitudes=True,
    #limit=20000
)

quakeml_path = os.path.join(output_dir, "comcat_redoubt_20090320_20090322.quakeml")
cat.write(quakeml_path, format="QUAKEML")
print(cat)

# Get station metadata

In [ ]:
irisclient = Client("IRIS")  # or "IRIS"

inv = irisclient.get_stations(
    latitude=REDOUBT_LAT, longitude=REDOUBT_LON,
    maxradius=STATION_RADIUS_DEG,
    starttime=t0, endtime=t1,
    level="response",
)

stationxml_path = os.path.join(output_dir, "Redoubt_20090320_20090322_station.xml")
inv.write(stationxml_path, format="STATIONXML")
print(inv)

# Get waveform data

In [ ]:
import os
from obspy import Stream

def write_stream_to_sds(
    st,
    sds_root,
    data_type="D",
    loc_blank_as="",
    merge=True,
    verbose=True,
):
    """
    NAIVE VERSION — for teaching purposes.

    Writes an ObsPy Stream to SDS directory structure with minimal
    preprocessing. This version intentionally assumes:

    - Sampling rates are identical
    - Data types are identical
    - No masked arrays
    - Merge will always succeed
    - No need to pad or trim to full days

    These assumptions are often violated in real-world data and will
    cause failures or multiple files per day.
    """

    st = st.copy()

    # Attempt simple merge (no normalization)
    if merge:
        try:
            st.merge(method=1, fill_value=None)
        except Exception as e:
            print(f"⚠️ Merge failed: {e}")

    # Split traces (may still leave multiple segments per day)
    out = Stream()
    for tr in st:
        try:
            out += tr.split()
        except Exception:
            out += tr
    st = out

    # Write each trace independently
    for tr in st:
        net = tr.stats.network
        sta = tr.stats.station
        loc = (tr.stats.location or "").strip()
        cha = tr.stats.channel

        if loc == "":
            loc = loc_blank_as

        t = tr.stats.starttime
        year = t.year
        jjj = f"{t.julday:03d}"

        # SDS directory path
        sds_dir = os.path.join(
            sds_root,
            f"{year}",
            net,
            sta,
            f"{cha}.{data_type}"
        )
        os.makedirs(sds_dir, exist_ok=True)

        # SDS filename
        fname = f"{net}.{sta}.{loc}.{cha}.{data_type}.{year}.{jjj}"
        full_path = os.path.join(sds_dir, fname)

        if verbose:
            print(f"Writing {full_path}")

        tr.write(full_path, format="MSEED")

In [ ]:
# Optional: restrict to common seismic channels
keep_prefixes = ("BH", "EH", "HH", "SH")  # broadband / short-period families

# 1) Build a list of channel codes to request (net, sta, loc, cha)
chans = []
for net in inv:
    for sta in net:
        for cha in sta:
            if not cha.code.startswith(keep_prefixes):
                continue
            loc = (cha.location_code or "").strip()
            chans.append((net.code, sta.code, loc, cha.code))

# De-duplicate (inventory sometimes repeats channels across epochs)
chans = sorted(set(chans))

# 2) Download and write daily files in SDS structure (no plots here)
day = t0
while day < t1:
    day_end = day + 24 * 3600

    st = irisclient.get_waveforms_bulk(
        bulk=[(n, s, l, c, day, day_end) for (n, s, l, c) in chans],
        attach_response=False,
    )

    # Write to SDS root (one file per channel per day, SDS naming)
    write_stream_to_sds(
        st,
        sds_root=output_dir,   # treat output_dir as SDS_ROOT
        data_type="D",
        loc_blank_as="",       # or "--" if you prefer
        merge=True,
        verbose=True,
    )

    day = day_end

print(f"Done. Wrote SDS daily files under: {output_dir}")


## Challenges Working with Real-World MiniSEED Archives

When working with continuous waveform archives from observatories and data centers, the data are rarely perfectly uniform. In this exercise we encountered several common issues that arise when assembling daily waveform files from many MiniSEED segments.

### 1. Gaps Between Segments

Continuous data are often stored as many small segments rather than a single uninterrupted file. These segments may:

- Have **small gaps** due to telemetry dropouts
- Have **overlaps** due to buffering or clock corrections
- Begin and end at arbitrary times rather than day boundaries

When merging segments, these gaps can prevent creation of a single continuous Trace unless we explicitly decide how to handle missing data.

---

### 2. Slightly Different Sampling Rates

Even when a channel is nominally sampled at a fixed rate (e.g., 100 Hz), we observed values such as:

- 99.999252 Hz  
- 99.999496 Hz  
- 100.000000 Hz  

These differences are typically caused by:

- Digitizer clock drift
- Rounding during archival
- Different firmware versions
- Floating-point representation

ObsPy considers traces with different sampling rates to be incompatible for merging, even when the difference is tiny.

---

### 3. Mixed Data Types

Different MiniSEED segments may store samples as:

- `int32`
- `float32`
- `float64`
- masked arrays

This can happen when:

- Data are processed differently before archiving
- Gaps are represented as masked arrays
- Instrument response corrections were applied to some segments but not others

Mixed data types can prevent merging or writing because ObsPy requires consistent numeric arrays.

---

### 4. Masked Arrays from Gap Handling

When gaps are present, some workflows create masked arrays internally.  
However, MiniSEED writing does **not support masked arrays**, leading to errors like:

NotImplementedError: Masked array writing is not supported

---

### 5. Multiple Segments Per Day

Because of the issues above, a single channel for one day may remain split into multiple traces after merging attempts, which violates the SDS convention of one file per channel per day.

---

## Why These Issues Matter

For long-term archives and reproducible workflows, we want:

- Consistent sampling rates
- Continuous daily files
- Predictable file structure
- Reliable downstream processing

Without normalization, downstream analysis (STA/LTA, spectral analysis, machine learning) can fail or produce inconsistent results.


⸻


## How Our New SDS Writing Functions Solve These Problems

To ensure robust and reproducible archive creation, we implemented a series of preprocessing steps before writing MiniSEED files to the SDS structure.

---

### 1. Sampling Rate Normalization

We round sampling rates to a consistent precision (e.g., 0.001 Hz) and resample traces when needed.

This allows traces like:

- 99.999252 Hz
- 99.999496 Hz
- 100.000000 Hz

to be treated as a single consistent sampling rate so they can be merged safely.

---

### 2. Data Type Standardization

All traces are converted to `float32`.

This ensures:

- Consistent numeric precision
- Compatibility during merge
- Stable MiniSEED writing
- Reduced storage size compared to float64

---

### 3. Masked Array Removal

Any masked arrays are converted to normal NumPy arrays using `.filled()`.

This prevents write errors and ensures gaps are handled explicitly rather than implicitly.

---

### 4. Controlled Gap Filling

We merge traces using:

merge(method=1, fill_value=0.0)

This produces a continuous waveform where gaps are filled with zeros, allowing exactly one trace per day.

This is consistent with many observatory archive practices when continuous coverage is required.

---

### 5. Enforcing Exact Day Boundaries

Each trace is padded and trimmed so that it spans exactly:

00:00:00 → 24:00:00 UTC

This guarantees:

- One file per channel per day
- Exact SDS compliance
- Predictable file sizes

---

### 6. Writing True SDS Layout

Files are written using the standard SeisComP SDS directory structure:

SDSROOT/YEAR/NET/STA/CHAN.TYPE/NET.STA.LOC.CHA.TYPE.YEAR.JJJ

This format is widely used by observatories and enables compatibility with tools like:

- SeisComP
- ObsPy SDS client
- Antelope
- Earthworm workflows

---

## Result

The new workflow produces a clean, standardized archive where each channel has:

✅ One MiniSEED file per day  
✅ Consistent sampling rate  
✅ Continuous waveform coverage  
✅ Stable numeric format  
✅ SDS-compliant directory structure  

This greatly improves reliability for downstream analysis and long-term archival.

In [ ]:
import os
import shutil

# Source and destination
src = os.path.join(output_dir, "2009")
dst = os.path.join(output_dir, "2009_fragmented")

print(f"Moving {src} -> {dst}")
shutil.move(src, dst)

In [ ]:
# import smarter version of the write_stream_to_sds function that handles more edge cases and is more robust for real-world data
from sds_utils import write_stream_to_sds

now re-run the previous cell:

In [ ]:
# Optional: restrict to common seismic channels
keep_prefixes = ("BH", "EH", "HH", "SH")  # broadband / short-period families

# 1) Build a list of channel codes to request (net, sta, loc, cha)
chans = []
for net in inv:
    for sta in net:
        for cha in sta:
            if not cha.code.startswith(keep_prefixes):
                continue
            loc = (cha.location_code or "").strip()
            chans.append((net.code, sta.code, loc, cha.code))

# De-duplicate (inventory sometimes repeats channels across epochs)
chans = sorted(set(chans))

# 2) Download and write daily files in SDS structure (no plots here)
day = t0
while day < t1:
    day_end = day + 24 * 3600

    st = irisclient.get_waveforms_bulk(
        bulk=[(n, s, l, c, day, day_end) for (n, s, l, c) in chans],
        attach_response=False,
    )

    # Write to SDS root (one file per channel per day, SDS naming)
    write_stream_to_sds(
        st,
        sds_root=output_dir,   # treat output_dir as SDS_ROOT
        data_type="D",
        loc_blank_as="",       # or "--" if you prefer
        merge=True,
        verbose=True,
    )

    day = day_end

print(f"Done. Wrote SDS daily files under: {output_dir}")

Finally, let's make a helicorder plot for each daily miniseed file:

In [ ]:
import os
from obspy import read

def make_dayplot_pngs_from_sds(
    sds_root,
    pattern_suffix=".D.",   # only plot waveform SDS files
    overwrite=False,
    verbose=True,
):
    """
    Walk an SDS tree and save a dayplot PNG next to each MiniSEED file.
    Output PNG path = <mseed_path>.png
    """
    n_total = 0
    n_written = 0
    n_skipped = 0
    n_failed = 0

    for root, _, files in os.walk(sds_root):
        for fn in files:
            # SDS files have no extension; skip obvious non-data
            if fn.endswith(".png"):
                continue
            if pattern_suffix not in fn:
                continue

            mseed_path = os.path.join(root, fn)
            png_path = mseed_path + ".png"

            n_total += 1

            if (not overwrite) and os.path.exists(png_path):
                n_skipped += 1
                continue

            try:
                st = read(mseed_path)

                # (Normally one trace per SDS file; but safe either way)
                for tr in st:
                    # Save figure; do not show
                    tr.plot(
                        type="dayplot",
                        outfile=png_path,
                        show=False
                    )
                    # If multiple traces exist, they'd overwrite the same png.
                    # We break to enforce 1 PNG per file.
                    break

                n_written += 1
                if verbose:
                    print(f"Wrote {png_path}")

            except Exception as e:
                n_failed += 1
                if verbose:
                    print(f"FAILED {mseed_path}: {e}")

    print(
        f"Dayplot PNGs: total={n_total}, written={n_written}, "
        f"skipped={n_skipped}, failed={n_failed}"
    )

make_dayplot_pngs_from_sds(
    sds_root=output_dir,
    overwrite=False,
    verbose=True,
)